In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import tensorflow_model_optimization as tfmot
import os
import numpy as np

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THÔNG SỐ
# ==========================================
BASE_PATH = "C:/DUT/Ki_2/Edge_AI/dataset/"
TRAIN_DIR = os.path.join(BASE_PATH, "train")

IMG_HEIGHT = 32
IMG_WIDTH = 32
BATCH_SIZE = 32
NUM_CLASSES = 10 # Tuỳ chỉnh theo số lượng biển báo của bạn

# ==========================================
# 2. CHUẨN BỊ DỮ LIỆU & AUGMENTATION
# ==========================================
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset="training", seed=42, 
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset="validation", seed=42, 
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE
)

# Chuẩn hóa (0-1) và Cache để train nhanh hơn
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(tf.data.AUTOTUNE)

# Data Augmentation (Tăng cường dữ liệu)
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# ==========================================
# 3. XÂY DỰNG & HUẤN LUYỆN MÔ HÌNH GỐC (BASE MODEL)
# ==========================================
print("\n--- PHÁT TRIỂN MÔ HÌNH GỐC ---")
base_model = models.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    data_augmentation,
    
    # Dùng SeparableConv2D để tối ưu SRAM (< 512KB)
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    
    layers.SeparableConv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.SeparableConv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

base_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train mô hình gốc khoảng 20-30 epochs (Giả định bạn đã train xong bước này)
# Ở đây mình để 10 epochs để demo nhanh
base_model.fit(train_ds, validation_data=val_ds, epochs=10)

# ==========================================
# 4. ÁP DỤNG QUANTIZATION-AWARE TRAINING (QAT)
# ==========================================
print("\n--- BẮT ĐẦU QUANTIZATION-AWARE TRAINING ---")
# Bọc toàn bộ base_model bằng hàm quantize_model
quantize_model = tfmot.quantization.keras.quantize_model

# QAT Model sinh ra các "Fake Quantization Nodes"
qat_model = quantize_model(base_model)

# Re-compile QAT model (BẮT BUỘC)
# Lưu ý: Lúc này phải dùng Learning Rate RẤT NHỎ (ví dụ: 1e-4 hoặc 1e-5)
qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

qat_model.summary()

# Huấn luyện tinh chỉnh (Fine-tune) QAT model khoảng 3-5 epochs
print("\nĐang tinh chỉnh (Fine-tuning) mô hình với QAT...")
qat_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[callbacks.ModelCheckpoint('best_qat_model.h5', save_best_only=True)]
)

# ==========================================
# 5. CHUYỂN ĐỔI SANG TFLITE (FULL INTEGER INT8)
# ==========================================
print("\n--- XUẤT MÔ HÌNH TFLITE TỪ QAT ---")

# Lấy Representative Dataset từ tập Train để calibrate độ lệch (zero_point/scale)
def representative_data_gen():
    for input_value, _ in train_ds.unbatch().batch(1).take(150):
        yield [input_value.numpy().astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Ràng buộc chạy full Int8 cho ESP32-S3
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_qat_model = converter.convert()

TFLITE_QAT_PATH = "traffic_sign_qat_int8.tflite"
with open(TFLITE_QAT_PATH, 'wb') as f:
    f.write(tflite_qat_model)

print(f"✅ Đã xuất thành công mô hình siêu tối ưu: {TFLITE_QAT_PATH}")
print(f"📏 Kích thước file: {os.path.getsize(TFLITE_QAT_PATH) / 1024:.2f} KB")

Found 2000 files belonging to 10 classes.
Using 1600 files for training.
Found 2000 files belonging to 10 classes.
Using 400 files for validation.

--- PHÁT TRIỂN MÔ HÌNH GỐC ---



Epoch 1/10
50/50 [==============================] - 5s 54ms/step - loss: 2.0358 - accuracy: 0.2625 - val_loss: 2.3026 - val_accuracy: 0.1175
Epoch 2/10
50/50 [==============================] - 3s 52ms/step - loss: 1.6466 - accuracy: 0.3775 - val_loss: 2.3049 - val_accuracy: 0.1175
Epoch 3/10
50/50 [==============================] - 2s 49ms/step - loss: 1.4476 - accuracy: 0.4512 - val_loss: 2.3161 - val_accuracy: 0.0925
Epoch 4/10
50/50 [==============================] - 2s 49ms/step - loss: 1.3454 - accuracy: 0.5075 - val_loss: 2.3282 - val_accuracy: 0.0925
Epoch 5/10
50/50 [==============================] - 2s 48ms/step - loss: 1.2289 - accuracy: 0.5412 - val_loss: 2.3560 - val_accuracy: 0.0925
Epoch 6/10
50/50 [==============================] - 2s 45ms/step - loss: 1.1039 - accuracy: 0.6037 - val_loss: 2.3886 - val_accuracy: 0.0925
Epoch 7/10
50/50 [==============================] - 2s 45ms/step - loss: 1.0276 - accuracy: 0.6369 - val_loss: 2.4044 - val_accuracy: 0.0925
Epoch 8/10
50

ValueError: Quantizing a keras Model inside another keras Model is not supported.